# Baseline v5.1 — PhoBERT Cross-Encoder (Vietnamese backbone)

## Điểm khác biệt so với v5

| | v5 (hiện tại) | **v5.1 (PhoBERT CE)** |
|--|--|--|
| CE backbone | `ms-marco-MiniLM-L-6-v2` (tiếng **Anh**) | `vinai/phobert-base` (tiếng **Việt**) |
| Pretrain corpus | MS MARCO (web search, EN) | 20GB văn bản tiếng Việt |
| Tokenizer | WordPiece (EN-biased) | BPE trên VN corpus |
| Mining config | SKIP_TOP_K=14, same | **SKIP_TOP_K=14, same** |
| Bi-encoder | v4 fine-tuned (giữ nguyên) | v4 fine-tuned (giữ nguyên) |

## Lý do dùng PhoBERT

MiniLM được pretrain trên **MS MARCO (tiếng Anh)** — tokenizer và language model chưa bao giờ học tiếng Việt thực sự.  
Dù đã fine-tune trên VN legal data, MiniLM vẫn tokenize từ tiếng Việt thành nhiều **subword lạ** và hiểu ngữ nghĩa kém hơn PhoBERT.

**PhoBERT** (`vinai/phobert-base`) được pretrain trên **20GB tiếng Việt** bằng RoBERTa framework — backbone hiểu tiếng Việt từ nền tảng, đặc biệt tốt với:
- Từ pháp lý VN: "khoản", "điều", "nghị định", "thông tư"
- Cấu trúc câu VN
- Dấu thanh và biến thể từ vựng

```
Pipeline v5.1:
  v4 bi-encoder (legal_hf_finetuned) + FAISS_V4
       ↓ retrieve top-30
  bỏ rank 1-14 (quá hard)
  lấy rank 15-30 làm medium-hard negatives
       ↓
  PhoBERT CE fine-tune trên hard_neg_train_v5_1.jsonl
       ↓
  Eval: kỳ vọng R@1 > 0.5418 (v5 MiniLM)
```

⚠️ **Lưu ý quan trọng:** PhoBERT dùng **BPE tokenizer riêng** và yêu cầu `sentencepiece`. Cần cài thêm:
```bash
pip install sentencepiece
```

| | v4_base | v4+CE(v5) | **v4+CE(v5.1)** |
|--|--|--|--|
| Recall@1 | 0.5232 | 0.5418 | ? |
| Recall@5 | 0.7337 | 0.7245 | ? |
| MRR@10   | 0.6091 | 0.6307 | ? |

## Cell 0 — Config & Imports

In [ ]:
# Cài sentencepiece nếu chưa có (PhoBERT yêu cầu)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=True)
print("sentencepiece ✓")

In [ ]:
import json, csv, time, random, gc
import numpy as np
import faiss
import torch
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

# ── Paths ──
ROOT          = Path(".")
DATA_DIR      = ROOT / "data"
EVAL_DIR      = ROOT / "outputs" / "eval"
TMP_DIR       = ROOT / "outputs" / "tmp"
MDL_DIR       = ROOT / "outputs" / "models"

TRAIN_FILE    = DATA_DIR / "train.jsonl"
DEV_FILE      = DATA_DIR / "dev.jsonl"
TRAIN_NEG     = DATA_DIR / "train_with_neg.jsonl"
EVAL_QA_FILE  = EVAL_DIR / "eval_qa.jsonl"

# ── v4 bi-encoder và FAISS (giữ nguyên từ v4) ──
FT_BI_PATH    = MDL_DIR  / "legal_hf_finetuned" / "final"
FAISS_V4      = TMP_DIR  / "faiss_v4.index"
MAP_V4        = TMP_DIR  / "faiss_mapping_v4.jsonl"

# ── Output paths v5.1 (TÁCH BIỆT với v5) ──
HN_TRAIN_FILE = EVAL_DIR / "hard_neg_train_v5_1.jsonl"   # hard neg mới cho training
CE_V51_DIR    = MDL_DIR  / "ce_phobert_v5_1"              # CE PhoBERT output
RERANK_CSV_V5 = EVAL_DIR / "rerank_metrics_v5.csv"        # v5 kết quả cũ (để so sánh)
RERANK_CSV_V51= EVAL_DIR / "rerank_metrics_v5_1.csv"      # v5.1 kết quả mới

# ── CE config: PhoBERT backbone ──
BASE_CE_MODEL = "vinai/phobert-base"   # ← THAY ĐỔI CHÍNH so với v5
CE_EPOCHS     = 5
CE_BATCH      = 16    # PhoBERT nặng hơn MiniLM → giảm batch để tránh OOM
CE_MAX_LEN    = 256
CE_LR         = 2e-5  # LR nhỏ cho PhoBERT
SEED          = 42

# ── Mining config (giống hệt v5) ──
TOP_MINE      = 30    # retrieve top-30
SKIP_TOP_K    = 14    # bỏ top-14 (vùng safe là rank 1-14)
HARD_NEG_PER  = 2     # tối đa 2 medium-hard negatives/query

TOP_N_EVAL    = 50
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED)
CE_V51_DIR.mkdir(parents=True, exist_ok=True)

print(f"torch  : {torch.__version__}")
print(f"Device : {DEVICE}")
print(f"CE Base: {BASE_CE_MODEL}  ← PhoBERT (tiếng Việt)")
print(f"Mining : top-{TOP_MINE}, skip top-{SKIP_TOP_K}, take {HARD_NEG_PER}/query")
print(f"  → rank {SKIP_TOP_K+1}→{TOP_MINE} làm medium-hard negatives")
print(f"Batch  : {CE_BATCH} (giảm từ 32 xuống 16 vì PhoBERT nặng hơn MiniLM)")

## Cell 1 — Utilities

In [ ]:
def load_jsonl(path, max_rows=None):
    rows, errors = [], 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows: break
            line = line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except json.JSONDecodeError: errors += 1
    if errors: print(f"  ⚠ {errors} malformed lines in {Path(path).name}")
    return rows

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

def is_hit(faiss_id, expected_citations, mapping):
    row = mapping[faiss_id]
    for ec in expected_citations:
        ci = ec.get("chunk_index", -2)
        if ci != -1 and row["chunk_index"] == ci: return True
        if (row["van_ban"] == ec.get("van_ban", "") and
            row["dieu"]    == ec.get("dieu",    "") and
            row["khoan"]   == ec.get("khoan",   "")): return True
    return False

def is_positive_meta(cand, meta):
    if cand["van_ban"] == meta.get("van_ban","") and cand["van_ban"] != "":
        if (cand["dieu"]  == meta.get("dieu","") and
            cand["khoan"] == meta.get("khoan","")): return True
    ci = meta.get("chunk_index", -2)
    if ci != -1 and cand["chunk_index"] == ci: return True
    return False

def avg(lst): return round(sum(lst)/len(lst), 4) if lst else 0.0

print("Utilities loaded ✓")

## Cell 2 — Load v4 Bi-Encoder + FAISS (giữ nguyên)

In [ ]:
print(f"Loading v4 fine-tuned bi-encoder: {FT_BI_PATH}")
ft_bi      = SentenceTransformer(str(FT_BI_PATH), device=DEVICE)
index_v4   = faiss.read_index(str(FAISS_V4))
mapping_v4 = load_jsonl(MAP_V4)
print(f"Bi-encoder ✓ | dim={ft_bi.get_sentence_embedding_dimension()}")
print(f"FAISS ✓ | {index_v4.ntotal} vectors, {len(mapping_v4)} mapping entries")

## Cell 3 — Medium-Hard Negative Mining (giống hệt v5)

```
Top-30 kết quả của v4 bi-encoder:

rank  1-14:  BỎ QUA  ← quá gần positive
rank 15-30:  LẤY LÀM NEGATIVE ← medium-hard
```
> Mining logic 100% giống v5 — chỉ CE backbone thay đổi ở Cell 5

In [ ]:
train_rows = load_jsonl(TRAIN_NEG)
pos_rows   = [r for r in train_rows if r.get("label") == 1]
random.seed(SEED); random.shuffle(pos_rows)
print(f"Positive rows: {len(pos_rows)}")

hn_train = []
stats    = {"pos": 0, "medium_hard": 0, "no_neg": 0}

for r in tqdm(pos_rows, desc="Mining medium-hard negatives"):
    query = r.get("query", "").strip()
    pos_p = r.get("passage", "").strip()
    meta  = r.get("meta", {})
    if not query or not pos_p: continue

    hn_train.append({"query": query, "passage": pos_p, "label": 1, "type": "positive"})
    stats["pos"] += 1

    q_emb  = ft_bi.encode([query], normalize_embeddings=True,
                           convert_to_numpy=True).astype("float32")
    _, ids = index_v4.search(q_emb, TOP_MINE)
    ids    = ids[0].tolist()

    medium_hard_pool = ids[SKIP_TOP_K:]   # rank 15-30

    added = 0
    for fid in medium_hard_pool:
        if fid < 0 or added >= HARD_NEG_PER: break
        cand = mapping_v4[fid]
        if cand["passage"] == pos_p: continue
        if is_positive_meta(cand, meta): continue

        hn_train.append({"query": query, "passage": cand["passage"],
                         "label": 0, "type": "medium_hard_neg",
                         "neg_rank": ids.index(fid) + 1})
        added += 1
        stats["medium_hard"] += 1

    if added == 0: stats["no_neg"] += 1

write_jsonl(HN_TRAIN_FILE, hn_train)
print(f"\n── Mining complete ──")
print(f"  Positive pairs    : {stats['pos']}")
print(f"  Medium-hard negs  : {stats['medium_hard']}  ({stats['medium_hard']/max(stats['pos'],1):.1f}/query avg)")
print(f"  No neg found      : {stats['no_neg']} queries")
print(f"  Total CE train    : {len(hn_train)} rows")
print(f"  Pos ratio         : {stats['pos']/len(hn_train):.1%}")
print(f"  Saved → {HN_TRAIN_FILE}")

## Cell 4 — Build Dev set với medium-hard negatives

In [ ]:
dev_rows = load_jsonl(DEV_FILE)
random.seed(SEED); random.shuffle(dev_rows)

hn_dev = []
for r in tqdm(dev_rows[:500], desc="Dev medium-hard negatives"):
    query = r.get("query", "").strip()
    pos_p = r.get("passage", "").strip()
    meta  = r.get("meta", {})
    if not query or not pos_p: continue

    hn_dev.append({"query": query, "passage": pos_p, "label": 1})

    q_emb  = ft_bi.encode([query], normalize_embeddings=True,
                           convert_to_numpy=True).astype("float32")
    _, ids = index_v4.search(q_emb, TOP_MINE)
    ids    = ids[0].tolist()

    for fid in ids[SKIP_TOP_K:]:       # rank 15-30
        if fid < 0: break
        cand = mapping_v4[fid]
        if cand["passage"] == pos_p: continue
        if is_positive_meta(cand, meta): continue
        hn_dev.append({"query": query, "passage": cand["passage"], "label": 0})
        break

random.seed(SEED); random.shuffle(hn_dev)
n_pos = sum(1 for r in hn_dev if r["label"]==1)
n_neg = sum(1 for r in hn_dev if r["label"]==0)
print(f"Dev set: {len(hn_dev)} rows | pos={n_pos}, neg={n_neg}")

## Cell 5 — Retrain Cross-Encoder với PhoBERT backbone

> ⏱️ Ước tính thời gian:
> - RTX 3050 Ti: ~**20-40 phút** (PhoBERT nặng hơn MiniLM ~2x)
> - Nếu OOM: giảm `CE_BATCH = 8` hoặc `CE_MAX_LEN = 128`
>
> 💡 PhoBERT có 135M params vs MiniLM 22M params → cần VRAM nhiều hơn

In [ ]:
# Giải phóng bi-encoder trước khi train CE
del ft_bi; gc.collect()
torch.cuda.empty_cache() if DEVICE == "cuda" else None
print("VRAM cleared ✓")

# Chuẩn bị samples
random.seed(SEED); random.shuffle(hn_train)
train_samples = [InputExample(texts=[r["query"], r["passage"]], label=float(r["label"]))
                 for r in hn_train if "query" in r and "passage" in r]
dev_samples   = [InputExample(texts=[r["query"], r["passage"]], label=float(r["label"]))
                 for r in hn_dev]

print(f"Train: {len(train_samples)} | Dev: {len(dev_samples)}")
print(f"Pos ratio: {sum(1 for s in train_samples if s.label==1)/len(train_samples):.1%}")

# ── LOAD PHOBERT làm CE backbone ──
# CrossEncoder tự thêm classification head (linear) lên PhoBERT
print(f"\nLoading PhoBERT CE: {BASE_CE_MODEL}")
print("(Lần đầu sẽ download ~509MB từ HuggingFace...)")
use_fp16  = (DEVICE == "cuda")
ce_v51    = CrossEncoder(
    BASE_CE_MODEL,
    num_labels=1,
    max_length=CE_MAX_LEN,
    device=DEVICE
)
evaluator = CEBinaryClassificationEvaluator.from_input_examples(dev_samples, name="dev_v5_1")
warmup    = int(len(train_samples) / CE_BATCH * CE_EPOCHS * 0.1)

print(f"\nTraining CE v5.1 (PhoBERT): {CE_EPOCHS} epochs, batch={CE_BATCH}, warmup={warmup}...")
t0 = time.time()

ce_v51.fit(
    train_dataloader=DataLoader(train_samples, shuffle=True, batch_size=CE_BATCH),
    evaluator=evaluator,
    epochs=CE_EPOCHS,
    warmup_steps=warmup,
    optimizer_params={"lr": CE_LR},
    output_path=str(CE_V51_DIR),
    use_amp=use_fp16,
)

elapsed = round((time.time()-t0)/60, 1)
print(f"Training done in {elapsed} min")

saved_path = CE_V51_DIR / "saved_model"
ce_v51.save(str(saved_path))

cfg = {
    "version": "v5.1",
    "base_model": BASE_CE_MODEL,
    "note": "PhoBERT backbone (VN pretrained) thay thế ms-marco-MiniLM (EN pretrained)",
    "epochs": CE_EPOCHS, "batch": CE_BATCH, "lr": CE_LR,
    "skip_top_k": SKIP_TOP_K, "top_mine": TOP_MINE, "hard_neg_per": HARD_NEG_PER,
    "train_samples": len(train_samples), "minutes": elapsed
}
(CE_V51_DIR / "config.json").write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"CE v5.1 (PhoBERT) saved → {saved_path}")

## Cell 6 — Evaluate v5.1: v4_FT_Bi + PhoBERT_CE

In [ ]:
# Reload bi-encoder
gc.collect()
torch.cuda.empty_cache() if DEVICE == "cuda" else None

print(f"Reloading v4 bi-encoder: {FT_BI_PATH}")
ft_bi      = SentenceTransformer(str(FT_BI_PATH), device=DEVICE)
index_v4   = faiss.read_index(str(FAISS_V4))
mapping_v4 = load_jsonl(MAP_V4)
eval_qa    = load_jsonl(EVAL_QA_FILE)
print(f"Loaded ✓ | Eval: {len(eval_qa)} questions")

r_base   = {"R@1":[], "R@3":[], "R@5":[], "MRR@10":[]}
r_rerank = {"R@1":[], "R@3":[], "R@5":[], "MRR@10":[]}

for item in tqdm(eval_qa, desc="Evaluate v5.1 (PhoBERT CE)"):
    query = item["query"]; ec = item["expected_citations"]
    q_emb = ft_bi.encode([query], normalize_embeddings=True,
                          convert_to_numpy=True).astype("float32")
    _, ids = index_v4.search(q_emb, TOP_N_EVAL)
    ids    = ids[0].tolist()

    # Baseline (v4)
    for k, key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_base[key].append(1 if any(is_hit(i,ec,mapping_v4) for i in ids[:k] if i>=0) else 0)
    mrr = 0.0
    for rank,i in enumerate(ids[:10],1):
        if i>=0 and is_hit(i,ec,mapping_v4): mrr=1.0/rank; break
    r_base["MRR@10"].append(mrr)

    # Rerank với PhoBERT CE v5.1
    cands   = [(mapping_v4[i]["passage"],i) for i in ids if i>=0]
    rscores = ce_v51.predict([[query,c[0]] for c in cands], batch_size=CE_BATCH) if cands else []
    ranked  = sorted(zip(rscores,[c[1] for c in cands]),reverse=True)
    r_ids   = [r[1] for r in ranked]

    for k, key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_rerank[key].append(1 if any(is_hit(i,ec,mapping_v4) for i in r_ids[:k]) else 0)
    mrr = 0.0
    for rank,i in enumerate(r_ids[:10],1):
        if is_hit(i,ec,mapping_v4): mrr=1.0/rank; break
    r_rerank["MRR@10"].append(mrr)

print("\n── v5.1 (PhoBERT CE) Results ──")
print(f"  {'Metric':<10} {'v4 Baseline':>14} {'v5.1 PhoBERT':>14} {'Δ':>8}")
print("  " + "-"*50)
for k, key in [("R@1","Recall@1"),("R@3","Recall@3"),("R@5","Recall@5"),("MRR@10","MRR@10")]:
    b  = avg(r_base[k]); re = avg(r_rerank[k])
    sign = "+" if re-b>=0 else ""
    icon = "✅" if re-b > 0 else ("=" if abs(re-b) < 0.001 else "❌")
    print(f"  {key:<10} {b:>14.4f} {re:>14.4f} {sign}{re-b:>7.4f} {icon}")

## Cell 7 — So sánh v5 (MiniLM) vs v5.1 (PhoBERT) & Lưu CSV

In [ ]:
# Đọc v5 kết quả cũ
v5 = {}
if RERANK_CSV_V5.exists():
    with open(RERANK_CSV_V5, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            v5[row["metric"]] = {
                "base":   float(row.get("v4_finetuned", 0) or 0),
                "rerank": float(row.get("v5_mh_reranked", 0) or row.get("v4_reranked", 0) or 0),
            }
else:
    # Fallback: hard-code kết quả v5 đã biết
    v5 = {
        "Recall@1": {"base": 0.5232, "rerank": 0.5418},
        "Recall@3": {"base": 0.6594, "rerank": 0.6873},
        "Recall@5": {"base": 0.7337, "rerank": 0.7245},
        "MRR@10":   {"base": 0.6091, "rerank": 0.6307},
    }
    print("⚠ rerank_metrics_v5.csv không tìm thấy → dùng giá trị cứng đã biết")

v51_base   = {"Recall@1":avg(r_base["R@1"]),   "Recall@3":avg(r_base["R@3"]),
              "Recall@5":avg(r_base["R@5"]),   "MRR@10":avg(r_base["MRR@10"])}
v51_rerank = {"Recall@1":avg(r_rerank["R@1"]), "Recall@3":avg(r_rerank["R@3"]),
              "Recall@5":avg(r_rerank["R@5"]), "MRR@10":avg(r_rerank["MRR@10"])}

print("\n" + "="*100)
print(f"  {'Metric':<10} {'v4_base':>12} {'v5+MiniLM':>12} {'v5.1+PhoBERT':>14} {'Δ(5.1-5)':>10}")
print("="*100)
for metric in ["Recall@1","Recall@3","Recall@5","MRR@10"]:
    v4b  = v5.get(metric, {}).get("base",  float("nan"))
    v5r  = v5.get(metric, {}).get("rerank",float("nan"))
    v51r = v51_rerank[metric]
    delta= v51r - v5r
    sign = "+" if delta >= 0 else ""
    icon = "✅" if delta > 0.001 else ("=" if abs(delta) <= 0.001 else "❌")
    print(f"  {metric:<10} {v4b:>12.4f} {v5r:>12.4f} {v51r:>14.4f} {sign}{delta:>9.4f} {icon}")
print("="*100)

# Lưu CSV
rows_v51 = [
    {"metric":"Recall@1", "v4_baseline":v5.get("Recall@1",{}).get("base",""), "v5_minilm":v5.get("Recall@1",{}).get("rerank",""), "v5_1_phobert":v51_rerank["Recall@1"]},
    {"metric":"Recall@3", "v4_baseline":v5.get("Recall@3",{}).get("base",""), "v5_minilm":v5.get("Recall@3",{}).get("rerank",""), "v5_1_phobert":v51_rerank["Recall@3"]},
    {"metric":"Recall@5", "v4_baseline":v5.get("Recall@5",{}).get("base",""), "v5_minilm":v5.get("Recall@5",{}).get("rerank",""), "v5_1_phobert":v51_rerank["Recall@5"]},
    {"metric":"MRR@10",   "v4_baseline":v5.get("MRR@10",{}).get("base",  ""), "v5_minilm":v5.get("MRR@10",{}).get("rerank",  ""), "v5_1_phobert":v51_rerank["MRR@10"]},
]
with open(RERANK_CSV_V51, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["metric","v4_baseline","v5_minilm","v5_1_phobert"])
    w.writeheader(); w.writerows(rows_v51)

print(f"\nSaved → {RERANK_CSV_V51} ✓")